In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import pandas  as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
viagem = pd.read_csv("/content/07-2025_pontos_do_esquema_operacional_sigma.csv", encoding='latin-1', sep=';', engine='python', on_bad_lines='skip')
viagem


FileNotFoundError: [Errno 2] No such file or directory: '/content/07-2025_pontos_do_esquema_operacional_sigma.csv'

**1º Passo: Limpeza de Dados**

* Analisar o dataset
* Limpar e analisar as colunas
* Remover colunas duplicadas

In [ ]:
# Analisando o número de linhas e colunas
viagem.shape
# Verificando os nomes das colunas
viagem.columns

**Verificando os valores nulos**

In [ ]:
# Analisando a quantidade de valores nulos presentes em nosso dataframe
viagem.isna().sum().sort_values()

**Verificando os valores únicos**

In [ ]:
# O metodo nunique retornar os valores únicos presentes em uma coluna
viagem.nunique().sort_values()

**Analisando os valores duplicados presentes nas colunas**

In [ ]:
# Utilizando o metodo duplicated para analisar a quantidade de valores duplicados presentes em nossas colunas
viagem.duplicated().sum()

**Fazendo uma cópia do nosso dataframe, isso é importante para não afetarmos o dataframe original ao realizarmos a análise**

In [ ]:
viagem_dados = viagem.copy()
viagem_dados.shape

**Limpando os valores nulos, verificamos anteriormente e agora iremos eliminá-los**

In [ ]:
# Limpando os valores nulos
viagem_dados = viagem_dados.dropna()
# Conferindo a quantidade de valores nulos após a limpeza, tem que ser 0
viagem_dados.isna().sum()

In [ ]:
# Podemos verificar que reduziu bastante com o shape
viagem_dados.shape

In [ ]:
# COR DOS GRÁFICOS
cor = ["darkblue", "black"]
sns.set_palette(cor)

**10 empresas com mais linhas**

In [ ]:
sns.set_theme(style="whitegrid")

rotas_por_empresa = viagem.groupby('razao_social')['descricao_linha'].nunique().sort_values(ascending=False)

plt.figure(figsize=(12, 6))
plt.style.use('seaborn-v0_8')

bars = plt.barh(
    rotas_por_empresa.head(10).index,
    rotas_por_empresa.head(10).values,
)

for bar in bars:
    bar.set_color('navy')

plt.title("Top 10 empresas com mais linhas cadastradas", fontsize=14, weight='bold')
plt.xlabel("Quantidade de linhas", fontsize=12)
plt.ylabel("Empresas", fontsize=12)

plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()



1.   **ANÁLISE**

sobre os ratings
1. Gontijo de transpotes limitada

2. Transportadora turistica suzano LTDA

3. Vianção aguia branca SA

O primeiro gráfico revela que que existé uma grande disparidade no número de linhas operadas entre as 3 maiores empresas com ênfase na número 1 (Gontijo de transportes limitada) possuindo praticamente a combinação da 2 e 3 posição sozinho.

Essa diferença acentuada revela um nível elevado de dominância operacional da Gontijo no setor analisado, sugerindo que ela possui:

maior cobertura territorial,

maior capacidade de frota,

e possivelmente uma demanda mais consolidada que suas concorrentes.



**10 empresas com maior presença em diferentes municípios**

In [ ]:
municipios_por_empresa = viagem.groupby('razao_social')['municipio_uf'].nunique().sort_values(ascending=False)

plt.figure(figsize=(12, 6))
plt.style.use('seaborn-v0_8')

bars = plt.barh(
    municipios_por_empresa.head(10).index,
    municipios_por_empresa.head(10).values,
)

for bar in bars:
    bar.set_color('navy')

plt.title("Top 10 empresas com presença em mais municípios", fontsize=14, weight='bold')
plt.xlabel("Número de municípios", fontsize=12)
plt.ylabel("Empresa", fontsize=12)

plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()


 2.   **ANÁLISE**

 sobre os ratings
 1. A Gontijo domina completamente. Ela aparece muito na frente das outras, atendendo cerca de 400 municípios
 2. Logo depois vem empresas como Guanabara, Suzano e Real Maia, com algo entre 230–280 municípios.
Elas têm uma presença forte, mas ainda assim bem distante da Gontijo.
  

**Distribuição dos veículos**

In [ ]:
tipos = viagem['tipo_veiculo'].value_counts().reset_index()
tipos.columns = ['tipo_veiculo', 'quantidade']

fig = px.bar(
    tipos,
    x='tipo_veiculo',
    y='quantidade',
    text='quantidade',
    title='Distribuição dos Tipos de Veículos',
    labels={'tipo_veiculo': 'Tipo de Veículo', 'quantidade': 'Quantidade'},
    color_discrete_sequence=['darkblue']
)

fig.update_traces(textposition='outside')

fig.update_layout(
    xaxis_title='Tipo de Veículo',
    yaxis_title='Quantidade',
    title_font_size=15,
    xaxis_tickfont_size=12,
    yaxis_tickfont_size=14,
    height=500
)

fig.show()

**10 Municípios com mais entradas de viagens**

In [ ]:
def extrair_municipio_destino(desc):
    try:
        partes = str(desc).split('-')
        if len(partes) >= 2:
            destino = partes[-1].strip()
            return destino
        else:
            return None
    except:
        return None


viagem['municipio_destino'] = viagem['descricao_linha'].apply(extrair_municipio_destino)

entradas_por_municipio = (
    viagem.groupby('municipio_destino')
    .size()
    .reset_index(name='quantidade_de_entradas')
    .sort_values(by='quantidade_de_entradas', ascending=False)
)

top_entradas = entradas_por_municipio.head(10)

fig = px.bar(
    top_entradas.sort_values(by='quantidade_de_entradas', ascending=True),
    x='quantidade_de_entradas',
    y='municipio_destino',
    color_discrete_sequence=['darkblue'],
    orientation='h',
    text='quantidade_de_entradas',
    title='Top 10 Distritos/Municípios com Mais Entradas de Viagens',
    labels={'municipio_destino': 'Município de Destino', 'quantidade_de_entradas': 'Número de Entradas'}
)

fig.update_traces(textposition='outside')
fig.update_layout(
    height=600,
    title_font_size=18,
    yaxis={'categoryorder': 'total ascending'}
)
fig.show()

**10 Municípios com mais saída**

In [ ]:
origem_municipios = viagem_dados[viagem_dados['sequencia_ponto_linha'] == 1]
top_municipios = origem_municipios['municipio_uf'].value_counts().reset_index()
top_municipios.columns = ['municipio_origem', 'quantidade_de_viagens']
top_municipios = top_municipios.head(10)

fig = px.bar(
    top_municipios,
    x='municipio_origem',
    y='quantidade_de_viagens',
    title='Top 10 Municípios com Mais Viagens de Saída',
    labels={'municipio_origem': 'Município de Origem', 'quantidade_de_viagens': 'Número de Viagens'},
    text='quantidade_de_viagens',
    color_discrete_sequence=['darkblue']

)
fig.update_traces(textposition='outside')
fig.update_layout(
    xaxis_tickangle=-30,
    xaxis_title_font_size=14,
    yaxis_title_font_size=14,
    title_font_size=18,
    height=500
)
fig.show()